# 00 · Helpers — Constantes e Funções Compartilhadas

> **Notebook compartilhado — não executar diretamente.**  
> Chamado via `%run 00_nb_helpers` pelos notebooks de transformação.

**Disponibiliza no namespace do chamador:**

| Símbolo | Tipo | Descrição |
|---|---|---|
| `CONCESSAO_MAP` | `Dict[str, str]` | Chave do filename → nome da concessionária |
| `COLS_VEICULOS` | `List[str]` | Colunas de contagem de veículos |
| `COLS_VITIMAS` | `List[str]` | Colunas de contagem de vítimas |
| `mapear_concessionaria()` | função | Deriva coluna `concessionaria` do filename |
| `transformar()` | função | Limpeza e padronização Bronze → Silver |
| `validar()` | função | Checagens de qualidade com log |
| `salvar_silver()` | função | Persiste Delta Table na camada Silver |

## Imports

In [ ]:
from functools import reduce
from typing import Dict, List

from pyspark.sql import Column, DataFrame, functions as F
from pyspark.sql.functions import when

## Constantes

In [ ]:
CONCESSAO_MAP: Dict[str, str] = {
    "aco":                  "RODOVIA DO AÇO",
    "af":                   "AUTOPISTA FLUMINENSE",
    "afd":                  "AUTOPISTA FERNÃO DIAS",
    "als":                  "AUTOPISTA LITORAL SUL",
    "aps":                  "AUTOPISTA PLANALTO SUL",
    "arb":                  "AUTOPISTA REGIS BITTENCOURT",
    "concebra":             "CONCEBRA",
    "concer":               "CONCER",
    "cro":                  "CRO",
    "eco050":               "ECO050",
    "eco101":               "ECO101",
    "ecoponte":             "ECOPONTE",
    "ecoriominas":          "ECORIOMINAS",
    "ecosul":               "ECOSUL",
    "ecoviasaraguaia":      "ECOVIAS DO ARAGUAIA",
    "ecoviascapixaba":      "ECOVIAS CAPIXABA",
    "ecoviasdocerrado":     "ECOVIAS DO CERRADO",
    "elovias":              "ELOVIAS",
    "epr_iguacu":           "EPR IGUACU",
    "epr_litoral_pioneiro": "EPR LITORAL PIONEIRO",
    "nova_381":             "NOVA 381",
    "pantanal":             "PANTANAL",
    "prvias":               "PRVIAS",
    "riosp":                "RIOSP",
    "rota-verde-goias":     "ROTA VERDE GOIÁS",
    "trans":                "TRANSBRASILIANA",
    "via040":               "VIA040",
    "viaaraucaria":         "VIA ARAUCARIA",
    "viabahia":             "VIABAHIA",
    "viabrasil":            "VIABRASIL",
    "viacosteira":          "VIACOSTEIRA",
    "viacristais":          "VIACRISTAIS",
    "viamineira":           "VIA MINEIRA",
    "viasul":               "VIASUL",
    "way_262":              "WAY 262",
}

COLS_VEICULOS: List[str] = [
    "automovel", "bicicleta", "caminhao", "moto", "onibus",
    "outros", "tracao_animal", "transporte_de_cargas_especiais",
    "trator_maquinas", "utilitarios",
]

COLS_VITIMAS: List[str] = [
    "ilesos", "levemente_feridos", "moderadamente_feridos",
    "gravemente_feridos", "mortos",
]

log.info("Constantes carregadas: %d concessionárias | %d cols_veiculos | %d cols_vitimas",
         len(CONCESSAO_MAP), len(COLS_VEICULOS), len(COLS_VITIMAS))

## Funções

In [ ]:
def ler_bronze(origem: str) -> DataFrame:
    """Lê todos os CSVs do caminho Bronze com encoding e separador corretos."""
    df = (
        spark.read
        .option("delimiter", ";")
        .option("header", "true")
        .option("encoding", "windows-1252")
        .option("quote", '"')
        .option("multiLine", "true")
        .csv(f"{origem}/*.csv")
        .withColumn("_filename", F.input_file_name())
    )
    log.info("Bronze lido: %d registros", df.count())
    return df


def _expr_concessionaria(mapa: Dict[str, str]) -> Column:
    """Constrói expressão when/otherwise para mapear chave do filename → concessionária."""
    keys = list(mapa.keys())
    expr = when(F.col("_file_key") == keys[0], mapa[keys[0]])
    for k in keys[1:]:
        expr = expr.when(F.col("_file_key") == k, mapa[k])
    return expr.otherwise("DESCONHECIDA")


def mapear_concessionaria(df: DataFrame, mapa: Dict[str, str]) -> DataFrame:
    """Extrai chave do filename e mapeia para nome da concessionária."""
    df = df.withColumn(
        "_file_key",
        F.regexp_extract(F.col("_filename"), r"demostrativo_acidentes_([^/\\.]+)\.csv", 1),
    ).withColumn("concessionaria", _expr_concessionaria(mapa))

    desconhecidas = df.filter(F.col("concessionaria") == "DESCONHECIDA").count()
    if desconhecidas > 0:
        log.warning("%d registros com concessionaria DESCONHECIDA — verifique novos arquivos", desconhecidas)

    return df.drop("_filename", "_file_key")


def transformar(df: DataFrame) -> DataFrame:
    """Aplica todas as transformações de limpeza e padronização Bronze → Silver."""

    # Data e hora
    df = (
        df
        .withColumn("data", F.to_date("data", "dd/MM/yyyy"))
        .withColumn("hora", F.split("horario", ":")[0].cast("int"))
    )

    # KM: vírgula → ponto → double
    df = df.withColumn("km", F.regexp_replace("km", ",", ".").cast("double"))

    # Colunas numéricas
    for col in COLS_VEICULOS + COLS_VITIMAS:
        df = df.withColumn(col, F.col(col).cast("int"))

    # Sentido: normaliza variações históricas
    df = df.withColumn(
        "sentido",
        when(F.upper(F.trim("sentido")).isin("NORTE", "N", "PISTA NORTE", "CRESCENTE"), "Norte")
        .when(F.upper(F.trim("sentido")).isin("SUL", "S", "PISTA SUL", "DECRESCENTE"), "Sul")
        .otherwise("Indefinido"),
    )

    # Tipo de ocorrência: normaliza 3 padrões históricos distintos
    df = df.withColumn(
        "tipo_de_ocorrencia",
        when(F.lower("tipo_de_ocorrencia").rlike(r"sem|s.vitima|ac01|ac02"), "Sem Vítima")
        .when(F.lower("tipo_de_ocorrencia").rlike(r"com|c.vitima|atropel|ac03"), "Com Vítima")
        .otherwise("Indefinido"),
    )

    # Trecho: separa rodovia e UF (ex: BR-116/SP)
    df = (
        df
        .withColumn("rodovia", F.trim(F.split("trecho", "/")[0]))
        .withColumn("uf",      F.trim(F.split("trecho", "/")[1]))
    )

    # Colunas derivadas de negócio
    soma_veiculos = reduce(lambda a, b: a + b, [F.coalesce(F.col(c), F.lit(0)) for c in COLS_VEICULOS])
    soma_vitimas  = reduce(lambda a, b: a + b, [F.coalesce(F.col(c), F.lit(0)) for c in COLS_VITIMAS])

    df = (
        df
        .withColumn("total_veiculos", soma_veiculos)
        .withColumn("total_vitimas",  soma_vitimas)
        .withColumn(
            "severidade",
            when(F.col("mortos") > 0,                 "Fatal")
            .when(F.col("gravemente_feridos") > 0,    "Grave")
            .when(F.col("moderadamente_feridos") > 0, "Moderado")
            .when(F.col("levemente_feridos") > 0,     "Leve")
            .otherwise("Sem Vítimas"),
        )
    )

    # Colunas de partição e análise temporal
    df = df.withColumn("ano", F.year("data")).withColumn("mes", F.month("data"))

    log.info("Transformações aplicadas.")
    return df


def validar(df: DataFrame) -> None:
    """Executa checagens de qualidade e loga os resultados."""
    campos_criticos = ["data", "km", "concessionaria", "rodovia", "uf", "tipo_de_acidente"]
    for campo in campos_criticos:
        nulos = df.filter(F.col(campo).isNull()).count()
        nivel = log.warning if nulos > 0 else log.info
        nivel("Nulos em %-35s %d", campo + ":", nulos)

    log.info("Registros por sentido:")
    for row in df.groupBy("sentido").count().orderBy("count", ascending=False).collect():
        log.info("  %-15s %d", row["sentido"], row["count"])

    log.info("Registros por severidade:")
    for row in df.groupBy("severidade").count().orderBy("count", ascending=False).collect():
        log.info("  %-15s %d", row["severidade"], row["count"])

    datas = df.agg(F.min("data").alias("min"), F.max("data").alias("max")).collect()[0]
    log.info("Range de datas: %s → %s", datas["min"], datas["max"])


def salvar_silver(df: DataFrame, tabela: str, modo: str, particoes: List[str]) -> None:
    """Persiste o DataFrame como Delta Table na camada Silver."""
    (
        df.write
        .format("delta")
        .mode(modo)
        .option("overwriteSchema", "true")
        .partitionBy(*particoes)
        .saveAsTable(tabela)
    )
    total = spark.sql(f"SELECT COUNT(*) AS total FROM {tabela}").collect()[0]["total"]
    log.info("Tabela %s salva: %d registros | partições: %s", tabela, total, particoes)


log.info("Helpers carregados.")